<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.1-diffusion-models/notebooks/exercises-4_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.1: Diffusion Models in Code

*Module 4 · Image, Vision & Multimodal AI*

> DALL-E, Stable Diffusion, Midjourney — they all work by starting with pure static and slowly removing noise until an image appears. Let's build this process from scratch in Python.

---

## About this notebook

This notebook contains the **8 runnable code examples** from the published lesson `lesson-4.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The Forward Process — Destroying Images with Noise — `part1_forward_process.py`
2. Step 2 — Noise Schedule — How MUCH Noise at Each Step? — `part2_noise_schedule.py`
3. Step 3 — The Denoising Network — Learning to Remove Noise — `part3_denoising_network.py`
4. Step 4 — The Training Loop — Teaching the Network to Denoise — `part4_training.py`
5. Step 5 — The Reverse Process — Creating Images from Nothing — `part5_reverse_process.py`
6. Step 6 — The Complete DDPM Pipeline — `part6_complete_ddpm.py`
7. Step 7 — Latent Diffusion & Text Conditioning — How Stable Diffusion Actually Works — `latent_diffusion.py`
8. Step 8 — Practical Generation — diffusers, Schedulers & ControlNet — `generate_images.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch numpy matplotlib accelerate


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The Forward Process — Destroying Images with Noise

Step 1 of understanding diffusion: learn how to gradually ruin a perfectly good image.

**`part1_forward_process.py`** · _Block 1 of 8_

FORWARD PROCESS: Gradually add noise to an image — This is how we CREATE the training data.


In [ ]:
# =============================================
# FORWARD PROCESS: Gradually add noise to an image
# This is how we CREATE the training data.
# Start: clean image. End: pure random noise.
# =============================================

import torch
import matplotlib.pyplot as plt
import numpy as np

# Create a simple "image" — a white square on black background
# In real diffusion models, this would be a photo loaded from disk
def make_sample_image(size=64):
    img = torch.zeros(1, size, size)  # 1 channel, 64x64
    img[0, 16:48, 16:48] = 1.0       # white square in center
    return img

image = make_sample_image()

# THE FORWARD PROCESS: add noise at each step
# At step t, the noisy image is:
#   x_t = sqrt(alpha_bar_t) * x_0  +  sqrt(1 - alpha_bar_t) * noise
#
# alpha_bar_t controls the MIX:
#   alpha_bar = 1.0 → 100% original image, 0% noise (clean)
#   alpha_bar = 0.5 → 50% image, 50% noise (half destroyed)
#   alpha_bar = 0.0 → 0% image, 100% noise (fully destroyed)

def add_noise(image, alpha_bar):
    """Mix the image with random noise according to alpha_bar."""
    noise = torch.randn_like(image)  # random noise, same shape as image
    
    noisy = torch.sqrt(alpha_bar) * image + torch.sqrt(1 - alpha_bar) * noise
    #        ↑ keep this much of image      ↑ add this much noise
    
    return noisy, noise

# Let's see what happens at different noise levels
alpha_bars = [0.99, 0.80, 0.50, 0.20, 0.05, 0.01]

fig, axes = plt.subplots(1, len(alpha_bars) + 1, figsize=(14, 2.5))

# Show original
axes[0].imshow(image[0], cmap="gray")
axes[0].set_title("Original", fontsize=9)
axes[0].axis("off")

# Show noisy versions
for i, ab in enumerate(alpha_bars):
    noisy, _ = add_noise(image, torch.tensor(ab))
    axes[i+1].imshow(noisy[0].clamp(0, 1), cmap="gray")
    axes[i+1].set_title(f"ᾱ={ab}", fontsize=9)
    axes[i+1].axis("off")

plt.suptitle("Forward Process: Gradually adding noise", fontsize=11)
plt.tight_layout()
plt.savefig("forward_process.png", dpi=100)
plt.show()

print("""
What you see:
  ᾱ = 0.99 → barely any noise (you can clearly see the square)
  ᾱ = 0.50 → half noise (square is fading)
  ᾱ = 0.05 → almost all noise (square is nearly invisible)
  ᾱ = 0.01 → pure static (no trace of the original image)

This is the forward process. We just learned to DESTROY images.
Now let's learn to UN-DESTROY them.
""")


> **What just happened?** We took a clean image (white square on black) and mixed it with random noise at 6 different levels. At ᾱ=0.99, the image is almost clean. At ᾱ=0.01, it's pure static. The formula is simple: noisy = (some % of original) + (some % of random noise) . The ᾱ value controls the mix. This is the "destroying" phase. The AI will learn to undo this destruction.


### Step 2 · Noise Schedule — How MUCH Noise at Each Step?

The schedule controls the pace of destruction. Too fast = model can't learn. Too slow = takes forever.

**`part2_noise_schedule.py`** · _Block 2 of 8_

NOISE SCHEDULE: Controls the pace of noising — β (beta) = how much noise to add at EACH step


In [ ]:
# =============================================
# NOISE SCHEDULE: Controls the pace of noising
# β (beta) = how much noise to add at EACH step
# ᾱ (alpha_bar) = how much TOTAL noise after t steps
# =============================================

import torch
import matplotlib.pyplot as plt

def linear_schedule(T=1000, beta_start=0.0001, beta_end=0.02):
    """Linear noise schedule — the original DDPM paper used this."""
    betas = torch.linspace(beta_start, beta_end, T)
    return betas

def cosine_schedule(T=1000):
    """Cosine schedule — smoother, used in improved DDPM."""
    steps = torch.arange(T + 1) / T
    alpha_bars = torch.cos((steps + 0.008) / 1.008 * torch.pi / 2) ** 2
    alpha_bars = alpha_bars / alpha_bars[0]
    betas = 1 - (alpha_bars[1:] / alpha_bars[:-1])
    return betas.clamp(max=0.999)

def compute_alpha_bars(betas):
    """From per-step betas, compute cumulative alpha_bars."""
    alphas = 1.0 - betas              # α_t = 1 - β_t
    alpha_bars = torch.cumprod(alphas, dim=0)  # ᾱ_t = α_1 × α_2 × ... × α_t
    return alpha_bars

# Compare the two schedules
T = 1000
linear_betas = linear_schedule(T)
cosine_betas = cosine_schedule(T)

linear_abars = compute_alpha_bars(linear_betas)
cosine_abars = compute_alpha_bars(cosine_betas)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(linear_betas.numpy(), label="Linear", color="#e11d48")
ax1.plot(cosine_betas.numpy(), label="Cosine", color="#0d9488")
ax1.set_title("β (noise added per step)")
ax1.set_xlabel("Timestep")
ax1.legend()

ax2.plot(linear_abars.numpy(), label="Linear", color="#e11d48")
ax2.plot(cosine_abars.numpy(), label="Cosine", color="#0d9488")
ax2.set_title("ᾱ (signal remaining after t steps)")
ax2.set_xlabel("Timestep")
ax2.legend()

plt.tight_layout()
plt.savefig("noise_schedules.png", dpi=100)
plt.show()

print(f"At step 0:   ᾱ = {linear_abars[0]:.4f} (almost clean)")
print(f"At step 500: ᾱ = {linear_abars[500]:.4f} (half destroyed)")
print(f"At step 999: ᾱ = {linear_abars[999]:.6f} (almost pure noise)")

print("""
Key insight:
  Linear schedule destroys the image too fast in the middle.
  Cosine schedule is gentler — the image fades more evenly.
  Modern models (Stable Diffusion) use cosine or custom schedules.
""")


> **What just happened?** We built two noise schedules: linear (original DDPM — adds noise steadily) and cosine (improved DDPM — smoother transition). The key number is ᾱ at each timestep: it tells you how much of the original image survives. At step 0, ᾱ≈1.0 (almost clean). At step 999, ᾱ≈0.0 (pure noise). The schedule controls the pace of this transition. Getting it right is critical for good image quality.


### Step 3 · The Denoising Network — Learning to Remove Noise

A neural network that looks at a noisy image and predicts what noise was added.

**`part3_denoising_network.py`** · _Block 3 of 8_

DENOISING NETWORK (simplified U-Net) — Input: noisy image + which timestep


In [ ]:
# =============================================
# DENOISING NETWORK (simplified U-Net)
# Input: noisy image + which timestep
# Output: predicted noise
#
# Real diffusion models use a full U-Net.
# We'll use a simplified version that captures
# the key idea in fewer lines.
# =============================================

import torch
import torch.nn as nn

class SimpleDenoiser(nn.Module):
    """
    Predicts the noise in a noisy image.
    
    Takes: noisy image (1, 64, 64) + timestep (integer)
    Returns: predicted noise (1, 64, 64) — same shape as input
    """
    
    def __init__(self, img_channels=1, hidden=64, T=1000):
        super().__init__()
        
        # Timestep embedding: convert step number → learned vector
        # The network needs to know HOW noisy the image is
        self.time_embed = nn.Sequential(
            nn.Embedding(T, hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        
        # Image processing layers (simplified U-Net)
        # Encoder: image → compressed representation
        self.encoder = nn.Sequential(
            nn.Conv2d(img_channels, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
        )
        
        # Middle: process with timestep info
        self.middle = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
        )
        
        # Decoder: back to image size → predicted noise
        self.decoder = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, img_channels, 3, padding=1),
        )
    
    def forward(self, x, t):
        # x shape: (batch, 1, 64, 64) — noisy image
        # t shape: (batch,) — timestep for each image
        
        # Get timestep embedding
        t_emb = self.time_embed(t)                # (batch, hidden)
        t_emb = t_emb.unsqueeze(-1).unsqueeze(-1)  # (batch, hidden, 1, 1)
        
        # Process image
        h = self.encoder(x)       # (batch, hidden, 64, 64)
        h = h + t_emb             # Add timestep info!
        h = self.middle(h)
        noise_pred = self.decoder(h)  # (batch, 1, 64, 64)
        
        return noise_pred

# Quick test
model = SimpleDenoiser()
fake_noisy = torch.randn(4, 1, 64, 64)  # 4 noisy images
fake_t = torch.randint(0, 1000, (4,))     # 4 random timesteps

predicted_noise = model(fake_noisy, fake_t)
print(f"Input:  {fake_noisy.shape}")         # (4, 1, 64, 64)
print(f"Output: {predicted_noise.shape}")    # (4, 1, 64, 64) — same!
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print("The network predicts noise — same shape as the input image.")


> **What just happened?** We built a neural network that takes a noisy image + a timestep number and outputs the predicted noise (same shape as the image). The timestep is crucial — it tells the network how noisy the image is, so it knows how much noise to predict. At early timesteps (little noise), it should predict small noise. At late timesteps (lots of noise), it should predict big noise. Real diffusion models use a full U-Net architecture (much bigger), but the concept is identical.


### Step 4 · The Training Loop — Teaching the Network to Denoise

Show it noisy images. Ask it to predict the noise. Tell it the right answer. Repeat.

**`part4_training.py`** · _Block 4 of 8_

TRAINING: Teach the network to predict noise — The simplest, most elegant training loop in AI.


In [ ]:
# =============================================
# TRAINING: Teach the network to predict noise
# The simplest, most elegant training loop in AI.
# =============================================

import torch
import torch.nn as nn

# Setup
T = 1000
betas = linear_schedule(T)
alpha_bars = compute_alpha_bars(betas)

model = SimpleDenoiser(T=T)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()  # Mean Squared Error between predicted and real noise

# Create a tiny dataset of simple shapes
def make_batch(batch_size=32):
    """Generate a batch of simple images (squares and circles)."""
    images = torch.zeros(batch_size, 1, 64, 64)
    for i in range(batch_size):
        # Random square position and size
        cx, cy = torch.randint(16, 48, (2,))
        s = torch.randint(8, 20, (1,)).item()
        images[i, 0, max(0,cx-s):min(64,cx+s), max(0,cy-s):min(64,cy+s)] = 1.0
    return images

# ── THE TRAINING LOOP ──
print("Training the denoiser...\n")

for epoch in range(100):
    # Step 1: Get clean images
    x_0 = make_batch(32)
    
    # Step 2: Pick random timesteps
    t = torch.randint(0, T, (32,))
    
    # Step 3: Add noise (forward process)
    ab = alpha_bars[t].view(-1, 1, 1, 1)   # reshape for broadcasting
    noise = torch.randn_like(x_0)             # the noise we add (this is the "answer")
    x_t = torch.sqrt(ab) * x_0 + torch.sqrt(1 - ab) * noise  # noisy image
    
    # Step 4: Network predicts the noise
    noise_pred = model(x_t, t)
    
    # Step 5: Compare predicted noise with real noise
    loss = loss_fn(noise_pred, noise)
    
    # Step 6: Update weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

print("\nTraining complete! The network learned to predict noise.")


> **What just happened?** The training loop is beautifully simple: (1) take a clean image, (2) add known random noise to it, (3) ask the network "what noise was added?", (4) compare the prediction with the real noise, (5) update the network to be more accurate. After 100 epochs, the loss drops — meaning the network is getting better at predicting noise. This is the entire DDPM training algorithm. The loss function is just MSE between predicted noise and actual noise. Elegant.


### Step 5 · The Reverse Process — Creating Images from Nothing

The magic moment: start with pure noise, remove it step by step, watch an image appear.

**`part5_reverse_process.py`** · _Block 5 of 8_

REVERSE PROCESS: Generate images from noise! — Start: pure random noise


In [ ]:
# =============================================
# REVERSE PROCESS: Generate images from noise!
# Start: pure random noise
# End: a clean image (hopefully!)
# =============================================

@torch.no_grad()
def generate(model, T=1000, img_size=64, save_every=100):
    """Generate an image by denoising pure noise step by step."""
    
    model.eval()
    betas = linear_schedule(T)
    alphas = 1.0 - betas
    alpha_bars = compute_alpha_bars(betas)
    
    # Start with pure random noise
    x = torch.randn(1, 1, img_size, img_size)
    
    snapshots = []  # save intermediate steps to see the process
    
    # Go from t=999 (noisy) to t=0 (clean), one step at a time
    for t_val in reversed(range(T)):
        t = torch.tensor([t_val])
        
        # Ask the network: "what noise is in this image?"
        predicted_noise = model(x, t)
        
        # Remove the predicted noise (the denoising formula)
        alpha = alphas[t_val]
        alpha_bar = alpha_bars[t_val]
        beta = betas[t_val]
        
        # Predict the clean image from the noisy one
        # x_0_pred = (x_t - sqrt(1-ᾱ) * predicted_noise) / sqrt(ᾱ)
        
        # Then take one denoising step
        if t_val > 0:
            noise = torch.randn_like(x)  # add a tiny bit of fresh noise (stochastic)
        else:
            noise = torch.zeros_like(x)  # last step: no fresh noise
        
        x = (1 / torch.sqrt(alpha)) * (
            x - (beta / torch.sqrt(1 - alpha_bar)) * predicted_noise
        ) + torch.sqrt(beta) * noise
        
        # Save snapshots
        if t_val % save_every == 0 or t_val == 0:
            snapshots.append((t_val, x.clone()))
    
    return x, snapshots

# ── Generate and visualize! ──
print("Generating image from pure noise...\n")
generated, snapshots = generate(model, T=1000, save_every=200)

# Show the generation process
fig, axes = plt.subplots(1, len(snapshots), figsize=(14, 2.5))
for i, (t, img) in enumerate(snapshots):
    axes[i].imshow(img[0, 0].clamp(0, 1), cmap="gray")
    axes[i].set_title(f"t={t}", fontsize=9)
    axes[i].axis("off")

plt.suptitle("Reverse Process: Image emerging from noise!", fontsize=11)
plt.tight_layout()
plt.savefig("reverse_process.png", dpi=100)
plt.show()

print("""
What you see:
  t=1000: pure random static
  t=800:  still mostly noise, maybe faint shapes
  t=400:  blurry shape emerging  
  t=200:  clearer shape visible
  t=0:    a generated image! (a square, since that's what we trained on)

You just generated an image FROM NOTHING.
The model started with random noise and created a shape
that looks like the training data — all by removing noise
one step at a time.
""")


> **What just happened?** We started with pure random noise and ran the reverse process: at each of 1,000 steps, the denoiser predicted "what noise is in this image?" and we subtracted it. Slowly, the noise faded and a shape appeared. Since we trained on white squares, the generated image is a white square. If we trained on faces, it would generate faces. If on cats, it would generate cats. The denoiser learned the "essence" of the training data and can recreate it from nothing but static. This is how DALL-E, Stable Diffusion, and Midjourney work.


### Step 6 · The Complete DDPM Pipeline

All 5 parts wrapped in one clean class you can reuse.

**`part6_complete_ddpm.py`** · _Block 6 of 8_

COMPLETE DDPM: Everything in one class. — Train on any dataset → Generate new images.


In [ ]:
# =============================================
# COMPLETE DDPM: Everything in one class.
# Train on any dataset → Generate new images.
# =============================================

class DDPM:
    """Denoising Diffusion Probabilistic Model — complete pipeline."""
    
    def __init__(self, img_channels=1, img_size=64, T=1000, lr=1e-3):
        self.T = T
        self.img_size = img_size
        
        # Noise schedule
        self.betas = linear_schedule(T)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = compute_alpha_bars(self.betas)
        
        # Denoising network
        self.model = SimpleDenoiser(img_channels, T=T)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()
    
    def train_step(self, x_0):
        """One training step: noise → predict → loss → update."""
        batch_size = x_0.size(0)
        
        # Random timesteps
        t = torch.randint(0, self.T, (batch_size,))
        
        # Forward: add noise
        ab = self.alpha_bars[t].view(-1, 1, 1, 1)
        noise = torch.randn_like(x_0)
        x_t = torch.sqrt(ab) * x_0 + torch.sqrt(1 - ab) * noise
        
        # Predict noise
        noise_pred = self.model(x_t, t)
        
        # Loss and update
        loss = self.loss_fn(noise_pred, noise)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    def train(self, make_batch_fn, epochs=200, batch_size=32):
        """Full training loop."""
        print(f"Training DDPM for {epochs} epochs...\n")
        for epoch in range(epochs):
            x_0 = make_batch_fn(batch_size)
            loss = self.train_step(x_0)
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1:3d} | Loss: {loss:.4f}")
        print("Training complete!")
    
    @torch.no_grad()
    def generate(self, num_images=4):
        """Generate new images from pure noise."""
        self.model.eval()
        x = torch.randn(num_images, 1, self.img_size, self.img_size)
        
        for t_val in reversed(range(self.T)):
            t = torch.full((num_images,), t_val, dtype=torch.long)
            predicted_noise = self.model(x, t)
            
            alpha = self.alphas[t_val]
            alpha_bar = self.alpha_bars[t_val]
            beta = self.betas[t_val]
            
            noise = torch.randn_like(x) if t_val > 0 else torch.zeros_like(x)
            x = (1 / torch.sqrt(alpha)) * (
                x - (beta / torch.sqrt(1 - alpha_bar)) * predicted_noise
            ) + torch.sqrt(beta) * noise
        
        self.model.train()
        return x.clamp(0, 1)

# ── Use it! ──
ddpm = DDPM(img_channels=1, img_size=64, T=500)

# Train on our simple shapes
ddpm.train(make_batch, epochs=200, batch_size=32)

# Generate new images!
print("\nGenerating 4 new images from pure noise...")
generated = ddpm.generate(num_images=4)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i in range(4):
    axes[i].imshow(generated[i, 0], cmap="gray")
    axes[i].set_title(f"Generated {i+1}", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Generated from pure noise by our DDPM!", fontsize=12)
plt.tight_layout()
plt.savefig("generated_images.png", dpi=100)
plt.show()

print("""
Your DDPM generated images from nothing but random static!
Each image shows a white square — because that's what it learned 
from training data. Train on faces → generates faces.
Train on landscapes → generates landscapes.

This is the EXACT same algorithm powering:
  • Stable Diffusion (+ text conditioning via CLIP)
  • DALL-E 2/3
  • Midjourney
  • Imagen (Google)
  
Same concept. Bigger models. Better training data.
""")


> **What just happened?** We wrapped everything into one clean DDPM class: schedule, model, training, and generation. Train it on any image dataset with .train() , then generate new images with .generate() . Our model generated white squares because that's what it trained on. DALL-E does the same thing but with: (a) a massive U-Net instead of our simple CNN, (b) 2 billion images instead of our 200 synthetic squares, (c) text conditioning (CLIP) that steers the generation toward "a cat wearing a hat." Same algorithm. Different scale.


### Step 7 · Latent Diffusion & Text Conditioning — How Stable Diffusion Actually Works

Run diffusion in compressed latent space (8× cheaper). Add text guidance via CLIP/T5 + cross-attention.

**`latent_diffusion.py`** · _Block 7 of 8_

LATENT DIFFUSION MODEL (LDM) Architecture


In [ ]:
# =============================================
# LATENT DIFFUSION MODEL (LDM) Architecture
# =============================================

# KEY INSIGHT: Run diffusion in LATENT space, not pixel space.
# A 512×512×3 image → VAE encoder → 64×64×4 latent → 64× fewer pixels!

architecture = {
    "VAE Encoder": "Image (512×512×3) → Latent (64×64×4). Compresses 8× spatially.",
    "VAE Decoder": "Latent (64×64×4) → Image (512×512×3). Reconstructs from latent.",
    "U-Net / DiT": "Denoises in latent space. Much faster than pixel-space diffusion.",
    "Text Encoder": "CLIP (SD 1.5/XL) or T5-XXL (SD 3/Flux). Converts prompt → embeddings.",
    "Cross-Attention": "Injects text embeddings INTO U-Net at every layer. Text guides denoising.",
}

# Classifier-Free Guidance (CFG) — the magic behind prompt following
# Run the model TWICE per step: once with prompt, once without.
# Amplify the difference: output = uncond + guidance_scale × (cond - uncond)
# guidance_scale=1.0 → ignores prompt. 7-12 → follows prompt. >15 → oversaturated.

def cfg_sample(model, x_t, t, text_emb, guidance_scale=7.5):
    # Predict noise WITH text
    noise_cond = model(x_t, t, text_embedding=text_emb)
    # Predict noise WITHOUT text (unconditional)
    noise_uncond = model(x_t, t, text_embedding=None)
    # Amplify the difference
    noise_guided = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
    return noise_guided

# Text Encoder Pipeline:
# SD 1.5: CLIP ViT-L/14 (77 tokens max, 768-dim)
# SDXL:   CLIP ViT-L/14 + OpenCLIP ViT-bigG (dual encoder)
# SD 3:   CLIP ViT-L + OpenCLIP ViT-bigG + T5-XXL (triple encoder!)
# Flux:   CLIP ViT-L + T5-XXL (dual encoder, flow matching)


> **What just happened?** Latent diffusion runs in compressed space — a 512×512 image becomes a 64×64 latent via VAE, making generation 8× cheaper. Text conditioning uses cross-attention : text embeddings from CLIP/T5 are injected into the U-Net at every layer, guiding the denoising direction. Classifier-Free Guidance (CFG) amplifies text adherence by computing the difference between conditioned and unconditioned predictions. Scale 7.5 is the sweet spot: lower ignores your prompt, higher causes artifacts. Modern models use multiple text encoders (SD 3 uses THREE: CLIP + OpenCLIP + T5) for better prompt understanding.


### Step 8 · Practical Generation — diffusers, Schedulers & ControlNet

Generate images in 5 lines of Python. Choose schedulers, control structure, and fine-tune with LoRA.

**`generate_images.py`** · _Block 8 of 8_

pip install diffusers transformers torch accelerate


In [ ]:
# pip install diffusers transformers torch accelerate
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

# Load SDXL (needs ~7GB VRAM)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# Use DPM++ 2M Karras scheduler (fastest for quality)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++",
    use_karras_sigmas=True,
)

# Generate!
image = pipe(
    prompt="A photorealistic portrait of an Indian classical dancer performing Bharatanatyam, "
           "golden temple background, dramatic lighting, 8k",
    negative_prompt="blurry, distorted, extra limbs, low quality",
    num_inference_steps=25,       # 20-30 with DPM++ is enough
    guidance_scale=7.5,           # CFG scale: 7-12 for photorealism
    width=1024, height=1024,     # SDXL native resolution
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]
image.save("bharatanatyam.png")

# ═══════ SCHEDULER COMPARISON ═══════
# DDPM:     1000 steps, highest quality, very slow
# DDIM:     50 steps, deterministic, good quality
# Euler:    20-30 steps, fast, default for many UIs
# DPM++ 2M: 20-25 steps, best speed-quality tradeoff ← RECOMMENDED
# UniPC:    10-15 steps, fastest, slight quality loss

# ═══════ CONTROLNET: Structure-guided generation ═══════
# from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
# controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny")
# → Input: edge map of desired composition
# → Output: image that follows the edges + your prompt
# Use cases: architectural viz, fashion design, product mockups

# ═══════ LoRA: Fine-tune for YOUR style ═══════
# pipe.load_lora_weights("path/to/lora/weights")
# 4-8 MB adapter (vs 6GB full model), trains in 30 min on 1 GPU
# Use cases: brand-specific style, character consistency, product photos

# ═══════ IP-Adapter: Image as prompt ═══════
# from diffusers import StableDiffusionXLPipeline
# pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models")
# → Input: reference image (e.g., your product photo)
# → Output: new images in the same style/subject
# Think: "LoRA = fine-tune style, IP-Adapter = transfer style at inference"

# ═══════ Production Optimization ═══════
# pipe.unet = torch.compile(pipe.unet)          # PyTorch 2.0+ → 20-30% faster
# pipe = pipe.to(torch_dtype=torch.float16)      # Half precision → 2× less VRAM
# TensorRT: NVIDIA's inference optimizer → 40-60% faster on NVIDIA GPUs
# Use ComfyUI (node-based) or Automatic1111 (web UI) for non-code workflows


> **What just happened?** Hugging Face diffusers makes generation production-ready in 5 lines. Key choices: DPM++ 2M Karras scheduler (25 steps, best speed-quality), guidance_scale=7.5 (prompt adherence), negative_prompt (tell the model what to avoid). ControlNet adds structural control (edge/depth/pose → guided composition). LoRA adapters fine-tune style in 4-8MB files (30 min, 1 GPU). IP-Adapter transfers style from a reference image at inference — no training needed. For production: torch.compile gives 20-30% speedup, TensorRT gives 40-60% on NVIDIA GPUs. Non-code workflows: ComfyUI (node-based, best for pipelines) or Automatic1111 (web UI, most popular).


---

## Wrap-up

You've walked through all 8 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
